# MNIST MLP3 — Muon horizon-optimized baseline

Runs a separate 30-epoch Muon + auxiliary AdamW candidate in `sgd_momentum_muon_horizon_optimized`. The recipe lowers the matrix peak LR and LR floor, strengthens matrix decay, and keeps Muon on `fc1.weight` and `fc2.weight` only. The official test set remains monitoring-only.

In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a current clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    MNIST_REFERENCE_RECIPE_VERSION,
    MNIST_REFERENCE_SUITE_SLUG,
    plot_all_replicates,
    run_baseline_replicates,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

BASE_RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
RUN_ROOT = BASE_RUN_ROOT / MNIST_REFERENCE_SUITE_SLUG
DATA_DIR = Path(os.environ.get('RG_BASELINE_DATA_DIR', ROOT / 'data')).expanduser().resolve()
for directory in (BASE_RUN_ROOT, RUN_ROOT, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('versioned suite root:', RUN_ROOT)


In [ ]:
CONFIG = BaselineConfig(
    optimizer='sgd_momentum_muon',
    epochs=30,
    validation_size=5_000,
    muon_parameter_names=('fc1.weight', 'fc2.weight'),
    muon_learning_rate=0.01,
    muon_min_learning_rate=2e-4,
    muon_warmup_epochs=2,
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_weight_decay=0.02,
    muon_newton_schulz_steps=5,
    muon_aux_learning_rate=3e-4,
    muon_aux_min_learning_rate=3e-6,
    muon_aux_beta1=0.90,
    muon_aux_beta2=0.95,
    muon_aux_weight_decay=0.01,
    ww_randomize=True,
    save_epoch_checkpoints=True,
)
CONFIG.validate()
if CONFIG.recipe_version != MNIST_REFERENCE_RECIPE_VERSION:
    raise RuntimeError('Notebook expects current MNIST recipe version.')
assert CONFIG.muon_parameter_names == ('fc1.weight', 'fc2.weight')
SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
RUN_DIR = RUN_ROOT / 'sgd_momentum_muon_horizon_optimized'
PLOT_DIR = RUN_DIR / 'plots'
for directory in (RUN_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('optimizer implementation:', CONFIG.optimizer_label)
print('horizon-optimized run directory:', RUN_DIR)
display(pd.DataFrame([CONFIG.__dict__]))


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
    resume=True,
    overwrite=False,
)
assert suite.config_template.recipe_version == MNIST_REFERENCE_RECIPE_VERSION
plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)


In [ ]:
metrics = [
    'train_loss', 'validation_loss', 'test_loss',
    'train_accuracy', 'validation_accuracy', 'test_accuracy',
    'validation_loss_gap', 'test_loss_gap',
    'validation_accuracy_gap', 'test_accuracy_gap',
    'primary_lr', 'auxiliary_lr', 'parameter_l2_norm',
    'mean_gradient_norm_before_clip', 'max_gradient_norm_before_clip',
]
for metric in metrics:
    if metric not in suite.performance.columns:
        continue
    summary = suite.performance_summary[suite.performance_summary['metric'].eq(metric)].sort_values('epoch')
    if summary.empty:
        continue
    figure, axis = plt.subplots(figsize=(9, 5))
    for seed, run in suite.performance.groupby('seed'):
        axis.plot(run['epoch'], run[metric], alpha=0.18, linewidth=0.8)
    axis.plot(summary['epoch'], summary['mean'], linewidth=2.0, label='mean')
    axis.fill_between(summary['epoch'], summary['ci_low'], summary['ci_high'], alpha=0.16)
    axis.set(xlabel='Epoch', ylabel=metric.replace('_', ' ').title(), title=f'{RUN_DIR.name}: {metric}')
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f'{metric}_95ci.png', dpi=170, bbox_inches='tight')
    plt.show()

display(suite.performance_summary[suite.performance_summary['metric'].isin(metrics)].sort_values(['metric', 'epoch']))


In [ ]:
required = [
    'alpha', 'ERG_gap', 'num_traps', 'm_midpoint',
    'trace_log_midpoint_per_eval', 'stable_rank',
    'normalized_lambda_max',
]
rows = suite.spectral_summary[suite.spectral_summary['metric'].isin(required)].sort_values(['metric', 'layer', 'epoch'])
assert rows['n'].eq(3).all()
display(rows)


In [ ]:
required_paths = [
    RUN_DIR / 'performance_by_epoch_and_seed.csv',
    RUN_DIR / 'spectral_metrics_by_epoch_layer_and_seed.csv',
    RUN_DIR / 'performance_summary_95ci.csv',
    RUN_DIR / 'spectral_summary_95ci.csv',
    RUN_DIR / 'replicate_manifest.json',
]
for seed in SEEDS:
    seed_dir = RUN_DIR / 'seeds' / f'seed_{seed}'
    required_paths.extend([
        seed_dir / 'checkpoint_latest.pt',
        seed_dir / 'checkpoint_best.pt',
        seed_dir / 'final_state.pt',
        seed_dir / 'test_results.json',
        seed_dir / 'run_complete.json',
    ])
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise RuntimeError('Missing required artifacts:' + chr(10) + chr(10).join(map(str, missing)))
print('verified artifacts:', len(required_paths))
